In [5]:
import pandas as pd
import os
import numpy as np
import re

raw_trait = pd.read_csv(os.path.join(os.getcwd(), "pgs_traits_data.csv"))
aou_gene_data = pd.read_csv("D:\Biostat_Research\Genetic_Agent\LOINC.csv")

In [6]:
contain_categories = ["Body measurement", "Cardiovascular measurement", "Hematological measurement", 
                      "Inflammatory measurement", "Lipid or lipoprotein measurement"]
pattern = "|".join(contain_categories)
raw_trait = raw_trait[raw_trait["Trait Category"].str.contains(pattern, na=False, regex=True)]

In [10]:
results = []
for idx, row in raw_trait.iterrows():
    ontology = str(row["Trait (ontology term label)"]).lower()
    words = re.findall(r'\b\w+\b', ontology)
    
    matched = aou_gene_data[
        aou_gene_data["description"]
        .apply(lambda x: all(word in str(x).lower() for word in words))
    ]
    print(matched, words)
    if not matched.empty:
        first_match = matched.iloc[0]
        loinc = first_match["loinc_code"]
        description = first_match["description"]
    else:
        loinc = np.nan
        description = np.nan
    
    results.append({
        "ontology": ontology,
        "pgs_num": row["Number of Related PGS"],
        "loinc": loinc,
        "description": description
    })

result_df = pd.DataFrame(results)

Empty DataFrame
Columns: [loinc_code, all_case_count, description]
Index: [] ['aortic', 'measurement']
Empty DataFrame
Columns: [loinc_code, all_case_count, description]
Index: [] ['apolipoprotein', 'a', '1', 'measurement']
Empty DataFrame
Columns: [loinc_code, all_case_count, description]
Index: [] ['apolipoprotein', 'a', 'iv', 'measurement']
Empty DataFrame
Columns: [loinc_code, all_case_count, description]
Index: [] ['apolipoprotein', 'b', 'measurement']
Empty DataFrame
Columns: [loinc_code, all_case_count, description]
Index: [] ['arterial', 'stiffness', 'measurement']
     loinc_code  all_case_count  \
31        706-2           90181   
40        704-7           85222   
126       707-0           14701   
222       705-4            6358   
351     12179-8            2552   
724     13519-4             490   
1412    57836-9              86   
1436    32154-7              83   
1816    32155-4              42   

                                            description  
31     Baso

In [11]:
import numpy as np

results = []

for idx, row in raw_trait.iterrows():
    ontology = str(row["Trait (ontology term label)"]).lower()
    words = re.findall(r'\b\w+\b', ontology)

    def score(desc):
        desc = str(desc).lower()
        return sum(word in desc for word in words) / max(len(words), 1)

    aou_gene_data["score"] = aou_gene_data["description"].apply(score)
    matched = aou_gene_data[aou_gene_data["score"] > 0]

    if not matched.empty:
        best = matched.sort_values("score", ascending=False).iloc[0]
        loinc = best["loinc_code"]
        description = best["description"]
    else:
        loinc = np.nan
        description = np.nan

    results.append({
        "ontology": ontology,
        "pgs_num": row["Number of Related PGS"],
        "loinc": loinc,
        "description": description
    })

result_df = pd.DataFrame(results)

In [12]:
print(result_df.shape)

(67, 4)


In [13]:
print(result_df)

                           ontology  pgs_num    loinc  \
0                aortic measurement        6      NaN   
1    apolipoprotein a 1 measurement        3   1871-3   
2   apolipoprotein a-iv measurement        1  14194-5   
3      apolipoprotein b measurement        4   1874-7   
4    arterial stiffness measurement        2   2019-8   
..                              ...      ...      ...   
62         triglyceride measurement       75   2571-8   
63            uric acid measurement        2  14628-2   
64     ventricular rate measurement        2   8867-4   
65                  vitamin d level       44  62291-0   
66                  waist-hip ratio       10  39156-5   

                                          description  
0                                                 NaN  
1   Apolipoprotein B-100 [Mass/volume] in Serum or...  
2        Spermatozoa Progressive/Spermatozoa in Semen  
3   Apolipoprotein B/Apolipoprotein A-I [Mass Rati...  
4   Carbon dioxide [Partial pressur

In [14]:
result_df.to_csv(os.path.join(os.getcwd(),"trait_list_260225.csv"), index=False)